### Import packages & load data

In [1]:
import pandas as pd

In [2]:
github_csv_url = 'https://raw.githubusercontent.com/sandrotissi/citibike_vrp/dcf382030b826796d3a5f94b5ddaa3d69c50b996/all_archetypes_demand_with_coordinates.csv'
df = pd.read_csv(github_csv_url)

print('Dataset loaded successfully. Displaying the first 5 rows:')
display(df.head())

Dataset loaded successfully. Displaying the first 5 rows:


,station_id,demand_00,demand_01,demand_10,demand_11,latitude,longitude
0,HB101,20.0,38.0,23.0,28.0,40.735938,-74.030305
1,HB102,0.0,23.0,0.0,20.0,40.736068,-74.029127
2,HB103,9.0,11.0,12.0,16.0,40.736982,-74.027781
3,HB105,12.0,18.0,10.0,17.0,40.737360,-74.030970
4,HB106,40.0,30.0,24.0,17.0,40.736722,-74.029007


### Distance Calculation using Google Maps API

As requested, we will now calculate the distances using the Google Maps Distance Matrix API for more accurate, real-world driving distances.

In [8]:
# Install the Google Maps client library
!pip install googlemaps

  Preparing metadata (setup.py) ... done
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40714 sha256=a6d032c0674d1467504d8028dbda4e400cea2ec4229d34d2f9fdfef7a33c9a21
  Stored in directory: /root/.cache/pip/wheels/4c/6a/a7/bbc6f5c200032025ee655deb5e163ce8594fa05e67d973aad6
Successfully built googlemaps


In [12]:
import googlemaps
from google.colab import userdata
import time
import numpy as np # Import numpy

# Retrieve API key from Colab secrets
GOOGLE_MAPS_API_KEY = userdata.get('GOOGLE_MAPS_API_KEY')
gmaps = googlemaps.Client(key=GOOGLE_MAPS_API_KEY)

In [13]:
def get_google_maps_distance(origin_lat, origin_lon, dest_lat, dest_lon):
    origins = f"{origin_lat},{origin_lon}"
    destinations = f"{dest_lat},{dest_lon}"

    try:
        # Request matrix
        matrix = gmaps.distance_matrix(origins, destinations, mode='driving')

        # Extract distance in meters and convert to kilometers
        # Check if the status is 'OK' and elements exist
        if matrix['status'] == 'OK' and matrix['rows'][0]['elements'][0]['status'] == 'OK':
            distance_meters = matrix['rows'][0]['elements'][0]['distance']['value']
            distance_km = distance_meters / 1000
            return distance_km
        else:
            print(f"Error in Google Maps API response: {matrix}")
            return np.nan
    except Exception as e:
        print(f"An error occurred: {e}")
        return np.nan

#### Calculate distance from depot to each station using Google Maps

Now, I'll use the Google Maps API to calculate the driving distance from the depot to each station.

In [14]:
# Depot coordinates
depot_latitude = 40.646420
depot_longitude = -74.016358

df['distance_from_depot_google_km'] = df.apply(lambda row: get_google_maps_distance(
    depot_latitude, depot_longitude,
    row['latitude'], row['longitude']
), axis=1)

print('Distances from depot to each station calculated using Google Maps. Displaying the first 5 rows with the new column:')
display(df.head())

Distances from depot to each station calculated using Google Maps. Displaying the first 5 rows with the new column:


,station_id,demand_00,demand_01,demand_10,demand_11,latitude,longitude,distance_from_depot_google_km
0,HB101,20.0,38.0,23.0,28.0,40.735938,-74.030305,17.969
1,HB102,0.0,23.0,0.0,20.0,40.736068,-74.029127,18.088
2,HB103,9.0,11.0,12.0,16.0,40.736982,-74.027781,18.305
3,HB105,12.0,18.0,10.0,17.0,40.737360,-74.030970,18.019
4,HB106,40.0,30.0,24.0,17.0,40.736722,-74.029007,18.161


#### Calculate distances between all pairs of stations using Google Maps

Finally, I'll create a distance matrix showing the Google Maps driving distance between every pair of stations. Due to API rate limits, this process might take some time for a large number of stations, and I'll include a small delay between requests.

In [16]:
num_stations = len(df)
station_distance_matrix_google_km = pd.DataFrame(np.zeros((num_stations, num_stations)),
                                               index=df['station_id'],
                                               columns=df['station_id'])

# Note: Google Maps API has usage limits. For many stations, this loop might take a long time
# or hit rate limits. Consider batching requests for production use.

for i in range(num_stations):
    for j in range(i + 1, num_stations):
        station1_lat, station1_lon = df.loc[i, ['latitude', 'longitude']]
        station2_lat, station2_lon = df.loc[j, ['latitude', 'longitude']]

        dist = get_google_maps_distance(station1_lat, station1_lon, station2_lat, station2_lon)
        station_distance_matrix_google_km.iloc[i, j] = dist
        station_distance_matrix_google_km.iloc[j, i] = dist # Matrix is symmetric
        time.sleep(0.1) # Small delay to avoid hitting rate limits too quickly

print('Station-to-station Google Maps distance matrix calculated. Displaying the top-left 5x5 sub-matrix:')
display(station_distance_matrix_google_km.head())

Station-to-station Google Maps distance matrix calculated. Displaying the top-left 5x5 sub-matrix:


station_id,HB101,HB102,HB103,HB105,HB106,HB201,HB202,HB203,HB301,HB302,...,JC134,JC135,JC137,JC138,JC139,JC140,JC142,JC143,JC144,JC145
station_id,,,,,,,,,,,,,,,,,,,,,
HB101,0.000,0.118,0.329,0.348,0.191,2.215,2.373,2.365,1.260,1.520,...,8.597,9.697,8.806,8.281,8.463,8.102,8.373,7.765,9.188,8.391
HB102,0.118,0.000,0.211,0.278,0.073,2.097,2.255,2.298,1.235,1.453,...,8.825,9.925,9.034,8.509,8.691,8.331,8.601,7.993,9.416,8.619
HB103,0.329,0.211,0.000,0.559,0.413,1.937,2.096,2.335,1.155,1.415,...,9.203,10.303,9.412,8.887,9.069,8.708,8.979,8.371,9.794,8.997
HB105,0.348,0.278,0.559,0.000,0.235,1.805,1.964,2.014,1.048,1.169,...,8.640,9.740,8.850,8.324,8.507,8.146,8.417,7.808,9.231,8.434
HB106,0.191,0.073,0.413,0.235,0.000,2.024,2.182,2.225,1.120,1.380,...,8.791,9.891,9.000,8.475,8.657,8.297,8.567,7.959,9.382,8.585


### Augmenting the Distance Matrix with Depot Distances

I will now add the depot as a point in the distance matrix. This involves creating a new row and column representing the distances from and to the depot for all stations, and a zero distance for the depot itself.

In [19]:
# Create a Series for depot distances to each station
depot_distances = df['distance_from_depot_google_km']
depot_distances.index = df['station_id']

# Add the depot to itself (distance 0)
depot_to_all = pd.Series([0] + depot_distances.tolist(), index=['depot'] + df['station_id'].tolist())

# Create a new row/column for the depot in the matrix
augmented_matrix_index = ['depot'] + station_distance_matrix_google_km.index.tolist()
augmented_matrix_columns = ['depot'] + station_distance_matrix_google_km.columns.tolist()

full_distance_matrix_google_km = pd.DataFrame(np.zeros((len(augmented_matrix_index), len(augmented_matrix_columns))),
                                                index=augmented_matrix_index,
                                                columns=augmented_matrix_columns)

# Fill in the station-to-station distances
full_distance_matrix_google_km.loc[station_distance_matrix_google_km.index, station_distance_matrix_google_km.columns] = station_distance_matrix_google_km

# Fill in the depot-to-station distances (row)
full_distance_matrix_google_km.loc['depot', full_distance_matrix_google_km.columns[1:]] = depot_distances.values

# Fill in the station-to-depot distances (column) - same as depot-to-station due to symmetry
full_distance_matrix_google_km.iloc[1:, full_distance_matrix_google_km.columns.get_loc('depot')] = depot_distances.values


print('Full distance matrix (including depot) calculated. Displaying the top-left 6x6 sub-matrix:')
display(full_distance_matrix_google_km.head(6).iloc[:, :6])

Full distance matrix (including depot) calculated. Displaying the top-left 6x6 sub-matrix:


,depot,HB101,HB102,HB103,HB105,HB106
depot,0.000,17.969,18.088,18.305,18.019,18.161
HB101,17.969,0.000,0.118,0.329,0.348,0.191
HB102,18.088,0.118,0.000,0.211,0.278,0.073
HB103,18.305,0.329,0.211,0.000,0.559,0.413
HB105,18.019,0.348,0.278,0.559,0.000,0.235
HB106,18.161,0.191,0.073,0.413,0.235,0.000


### Saving the Distance Matrix

I will now save the `full_distance_matrix_google_km` to a CSV file named `full_distance_matrix_google.csv`. You can then easily import this document into other applications.

In [20]:
output_filename = 'full_distance_matrix_google.csv'
full_distance_matrix_google_km.to_csv(output_filename, index=True)
print(f"The full distance matrix has been saved to '{output_filename}'")

The full distance matrix has been saved to 'full_distance_matrix_google.csv'
